# Fraud detection for Malaysia XX Bank: in-database XGBoost
Adapted from `UseCases/Financial_Fraud_Detection_InDB`. Same flow (feature prep, `TrainTestSplit`, in-database `XGBoost`, `XGBoostPredict`, `ClassificationEvaluator`, `ROC`), but on the synthetic 200k-row Malaysia XX Bank transaction table (`txn`, loaded by `scripts/load_data.py`) with the `txn_features` view.

**Outputs used by the FastAPI service:** tables `fraud_xgb_model`, `txn_scores`, `model_metrics`.

## 1. Connect

In [1]:
import os, time, getpass
from teradataml import *
import pandas as pd

# Password comes from the TD_PASSWORD env var (non-interactive runs), else prompt.
password = os.environ.get("TD_PASSWORD") or getpass.getpass("Teradata password: ")
eng = create_context(host=os.environ.get("TD_HOST", "host.docker.internal"),
                     username=os.environ.get("TD_USER", "demo_user"), password=password)
print(execute_sql("SELECT InfoData FROM DBC.DBCInfoV WHERE InfoKey='VERSION'").fetchall())

[['20.00.24.60']]


## 2. Features (computed in-database by the `txn_features` view)

In [2]:
FEATS = ["amount_myr", "is_foreign", "device_new", "hour_of_day", "km_from_home", "txn_count_1h",
         "amt_ratio_30d", "account_age_days", "ch_ecom", "ch_duitnow", "ch_fpx", "ch_atm",
         "mc_highrisk", "mc_resale", "night_txn"]
tdf = DataFrame("txn_features")
print(tdf.shape)
tdf.head(3)

(200000, 17)


txn_id,is_fraud,amount_myr,is_foreign,device_new,hour_of_day,km_from_home,txn_count_1h,amt_ratio_30d,account_age_days,ch_ecom,ch_duitnow,ch_fpx,ch_atm,mc_highrisk,mc_resale,night_txn
3,0,63.27,0,0,6,8.9,2,1.58,3607,0,0,1,0,0,1,0
2,0,36.63,0,0,12,9.7,0,0.91,279,1,0,0,0,0,0,0
1,0,43.13,0,0,1,4.4,0,0.42,988,0,1,0,0,0,0,1


In [3]:
# Class balance, computed in-DB
execute_sql("SELECT is_fraud, COUNT(*) FROM txn_features GROUP BY 1").fetchall()

[[1, 2823], [0, 197177]]

## 3. Train / test split (80/20, seed 42)

In [4]:
split = TrainTestSplit(data=tdf, id_column="txn_id", train_size=0.8, test_size=0.2, seed=42)
train = split.result[split.result.TD_IsTrainRow == 1].drop("TD_IsTrainRow", axis=1)
test = split.result[split.result.TD_IsTrainRow == 0].drop("TD_IsTrainRow", axis=1)
print("train", train.shape[0], "test", test.shape[0])

train 160000 test 40000


## 4. Train XGBoost inside Teradata

In [5]:
t0 = time.time()
model = XGBoost(data=train, input_columns=FEATS, response_column="is_fraud",
                model_type="Classification", iter_num=50, max_depth=6, lambda1=1,
                shrinkage_factor=0.3, min_node_size=5, base_score=0.0141,
                num_boosted_trees=8, seed=42)  # base_score = fraud base rate
train_secs = round(time.time() - t0, 1)
print("trained in-database in", train_secs, "seconds")
model.result.to_sql("fraud_xgb_model", if_exists="replace")  # persisted model table

trained in-database in 5.2 seconds


## 5. Evaluate on the held-out test set

In [6]:
pred = XGBoostPredict(newdata=test, object=model.result, id_column="txn_id",
                      model_type="Classification", output_prob=True, output_responses=["0", "1"],
                      accumulate="is_fraud")
ev = ClassificationEvaluator(data=pred.result, observation_column="is_fraud",
                             prediction_column="Prediction", labels=["0", "1"])
print(ev.output_data)

   SeqNum              Metric  MetricValue
0       3        Micro-Recall     0.952225
1       5     Macro-Precision     0.583831
2       6        Macro-Recall     0.783800
3       7            Macro-F1     0.622818
4       9     Weighted-Recall     0.952225
5      10         Weighted-F1     0.965082
6       8  Weighted-Precision     0.982150
7       4            Micro-F1     0.952225
8       2     Micro-Precision     0.952225
9       1            Accuracy     0.952225


In [7]:
roc = ROC(data=pred.result, probability_column="Prob_1", observation_column="is_fraud",
          positive_class="1")
roc.result

AUC,GINI
0.924058000489862,0.848116000979724


## 6. Score every transaction in-database and persist (dashboard and alerts read this)

In [8]:
allpred = XGBoostPredict(newdata=DataFrame("txn_features"), object=DataFrame("fraud_xgb_model"),
                         id_column="txn_id", model_type="Classification", output_prob=True,
                         output_responses=["0", "1"])
allpred.result.to_sql("txn_scores", if_exists="replace", primary_index="txn_id")
print(allpred.result.show_query()[:600])  # the SQL Teradata ran, shown in the app's "Show SQL" panel

select * from "DEMO_USER"."ml__td_sqlmr_out__179148687642440"


## 7. Persist metrics for the app's model card

In [9]:
auc_row = roc.result.to_pandas().iloc[0]
metrics = pd.DataFrame([{"auc": float(auc_row["AUC"]), "gini": float(auc_row["GINI"]),
    "train_rows": int(train.shape[0]), "test_rows": int(test.shape[0]), "train_seconds": train_secs,
    "algorithm": "Teradata in-database XGBoost (TD_XGBoost)", "features": ",".join(FEATS)}])
copy_to_sql(metrics, "model_metrics", if_exists="replace")
metrics

,auc,gini,train_rows,test_rows,train_seconds,algorithm,features
0,0.924058,0.848116,160000,40000,5.2,Teradata in-database XGBoost (TD_XGBoost),"amount_myr,is_foreign,device_new,hour_of_day,k..."


In [10]:
remove_context()

True